In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from multiprocessing import Pool

LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def moves(s):
    return [i for i in range(9) if s[i] == 0]

def play(s, a):
    s = s.copy()
    s[a] = 1
    return -s

def result(s):
    for a, b, c in LINES:
        if s[a] == s[b] == s[c] == -1:
            return -1.0
    return 0.0 if 0 not in s else None

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(9, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        self.pi = nn.Linear(64, 9)
        self.v = nn.Linear(64, 1)

    def forward(self, x):
        h = self.body(x)
        return self.pi(h), torch.tanh(self.v(h)).squeeze(-1)

def predict(net, s):
    with torch.no_grad():
        logits, v = net(torch.tensor(s).float().unsqueeze(0))
        p = F.softmax(logits, 1)[0].numpy()

    mask = np.zeros(9)
    mask[moves(s)] = 1
    p = p * mask

    return p / p.sum(), v.item()

class Node:
    def __init__(self, p):
        self.p = p
        self.n = 0
        self.w = 0.0
        self.kids = {}

def expand(net, node, s):
    p, v = predict(net, s)

    for a in moves(s):
        node.kids[a] = Node(p[a])

    return v

def mcts(net, s0, sims=50, c=1.5):
    root = Node(1)
    expand(net, root, s0)

    for _ in range(sims):
        node, s, path = root, s0, [root]

        while node.kids:
            N = math.sqrt(node.n + 1)

            a, node = max(
                node.kids.items(),
                key=lambda kv:
                    (-kv[1].w / kv[1].n if kv[1].n else 0)
                    + c * kv[1].p * N / (1 + kv[1].n)
            )

            s = play(s, a)
            path.append(node)

        v = result(s)

        if v is None:
            v = expand(net, node, s)

        for n in reversed(path):
            n.n += 1
            n.w += v
            v = -v

    pi = np.zeros(9)

    for a, k in root.kids.items():
        pi[a] = k.n

    return pi / pi.sum()

def self_play(net, sims=50):
    s = np.zeros(9, dtype=np.float32)
    hist = []

    while True:
        pi = mcts(net, s, sims)
        hist.append((s.copy(), pi))
        s = play(s, np.random.choice(9, p=pi))

        r = result(s)

        if r is not None:
            break

    return [
        (st, p, -r if i % 2 == 0 else r)
        for i, (st, p) in enumerate(reversed(hist))
    ]

def worker(args):
    state_dict, n_games = args

    torch.set_num_threads(1)

    net = Net()
    net.load_state_dict(state_dict)

    return [
        ex
        for _ in range(n_games)
        for ex in self_play(net)
    ]

def train(net, data, steps=50):
    opt = torch.optim.Adam(net.parameters(), 1e-3)

    S = torch.tensor(
        np.array([d[0] for d in data])
    ).float()

    P = torch.tensor(
        np.array([d[1] for d in data])
    ).float()

    Z = torch.tensor(
        [d[2] for d in data]
    ).float()

    for _ in range(steps):
        logits, v = net(S)

        loss = (
            -(P * F.log_softmax(logits, 1)).sum(1).mean()
            + F.mse_loss(v, Z)
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

    return loss.item()

if __name__ == "__main__":
    net = Net()
    buffer = []

    with Pool(4) as pool:
        for it in range(10):
            jobs = [
                (net.state_dict(), 8)
                for _ in range(4)
            ]

            for games in pool.map(worker, jobs):
                buffer += games

            buffer = buffer[-5000:]

            print(
                f"iter {it + 1}: "
                f"examples={len(buffer)}, "
                f"loss={train(net, buffer):.3f}"
            )

iter 1: examples=216, loss=2.708
iter 2: examples=443, loss=2.592
iter 3: examples=666, loss=2.438
iter 4: examples=892, loss=2.246
iter 5: examples=1119, loss=2.075
iter 6: examples=1363, loss=1.909
iter 7: examples=1611, loss=1.793
iter 8: examples=1851, loss=1.696
iter 9: examples=2110, loss=1.618
iter 10: examples=2375, loss=1.557
